# Checking how to enable PyApprox functionality in SPAROW

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pyapprox.util.backends.numpy import NumpyBkd
from pyapprox_benchmarks.statest import (
    PolynomialEnsembleBenchmark,
)
from pyapprox.statest.statistics import MultiOutputMean
from pyapprox.statest.mc_estimator import MCEstimator
from pyapprox.statest import (
    MLMCEstimator, MFMCEstimator, GMFEstimator, GRDEstimator, GISEstimator,
)
from pyapprox.statest.acv import ACVAllocator, default_allocator_factory
from pyapprox.statest.acv.base import FittedACVEstimator
from pyapprox.statest.acv.search import ACVSearch
from pyapprox.statest.acv.strategies import (
    FixedRecursionStrategy, TreeDepthRecursionStrategy,
)
from pyapprox.statest.allocation import MCAllocator, CVAllocator
from pyapprox.statest.plotting import (
    plot_allocation, plot_estimator_variance_reductions,
)
from pyapprox.optimization.minimize.scipy.slsqp import ScipySLSQPOptimizer

from sparow.ci.cli import load_problem_adapter, load_scenarios, load_xhat
from sparow.ci.pyapprox_interface import (
    convert_pyapprox_allocation_to_acvmrp_params, build_pyapprox_mf_problem_from_adapter
)

In [3]:
bkd = NumpyBkd()
np.random.seed(42)

# SLSQP is more robust than the default trust-constr optimizer for ACV allocation
optimizer = ScipySLSQPOptimizer(maxiter=200)
allocator_factory = lambda est: default_allocator_factory(est, optimizer=optimizer)

# ------------------------------------------------------
# User settings
# ------------------------------------------------------
MODEL_MODULE = "sparow_examples.mrp_facilityloc.mrp_discrete_facilityloc"
MODEL_NAME = "HF"
SCENARIO_FILE = "discrete_facilityloc_scenarios.npy"
XHAT_FILE = "../../../sparow/sparow/ci/manually_created_suboptimal_xhat.npy"

BATCH_SIZE = 100
SOLVER_NAME = "gurobi_direct"
SEED = 12345

# ------------------------------------------------------
# Load adapter, scenarios, and candidate solution
# ------------------------------------------------------
problem_adapter = load_problem_adapter(
    model_module_name=MODEL_MODULE,
    model_name=MODEL_NAME,
    use_integer=False,
    lf_model_type="classic",
)

full_scenarios = load_scenarios(SCENARIO_FILE)
xhat = load_xhat(XHAT_FILE)
if "ROOT" in xhat:
    xhat = xhat["ROOT"]

# ------------------------------------------------------
# Build PyApprox multifidelity problem
# ------------------------------------------------------
problem, bkd = build_pyapprox_mf_problem_from_adapter(
    problem_adapter=problem_adapter,
    full_scenarios=full_scenarios,
    xhat=xhat,
    batch_size=BATCH_SIZE,
    solver_name=SOLVER_NAME,
    solver_options=None,
    seed=SEED,
)

models = problem.models()
variable = problem.prior()
costs = problem.costs()
nqoi = models[0].nqoi()
nmodels = len(models)

costs_np = bkd.to_numpy(costs)
print(f"{nmodels} models, {nqoi} QoI(s)")
for a, c in enumerate(costs_np):
    print(f"  model {a}: estimated cost = {c:.6f}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveF

2 models, 1 QoI(s)
  model 0: estimated cost = 3.281131
  model 1: estimated cost = 2.571423


## Stage 1: Pilot Study

In [4]:
N_pilot = 100
samples_pilot = variable.rvs(N_pilot)
vals_pilot = [m(samples_pilot) for m in models]

stat = MultiOutputMean(nqoi, bkd)
cov_pilot, = stat.compute_pilot_quantities(vals_pilot)
stat.set_pilot_quantities(cov_pilot)

# Inspect pilot correlations with HF model
cov_np = bkd.to_numpy(cov_pilot)
for a in range(1, nmodels):
    rho = cov_np[0, a] / np.sqrt(cov_np[0, 0] * cov_np[a, a])
    print(f"  Pilot correlation ρ(f0, f{a}) = {rho:.4f}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveF

  Pilot correlation ρ(f0, f1) = 0.9935


In [6]:
total_budget = 1000.0
pilot_cost   = float(costs_np.sum()) * N_pilot
remaining    = total_budget - pilot_cost
print(f"Pilot cost: {pilot_cost:.1f}  |  Remaining: {remaining:.1f}")

Pilot cost: 585.3  |  Remaining: 414.7


## Stage 2: Build Estimator and Allocate

In [7]:
est = MFMCEstimator(stat, costs)
allocator = default_allocator_factory(est)
result = allocator.allocate(remaining)
fitted = FittedACVEstimator(est, result)

print(f"Samples per model: {fitted.nsamples_per_model()}")
print(f"Predicted std:     {float(fitted.covariance()[0,0])**0.5:.6f}")

Samples per model: [ 14 142]
Predicted std:     12395.703633


## Stage 3: Generate Samples

In [8]:
samples_per_model = fitted.generate_samples_per_model(variable.rvs)
print(f"Sample shapes: {[s.shape for s in samples_per_model]}")

Sample shapes: [(400, 14), (400, 142)]


## Stage 4: Evaluate Models

In [ ]:
values_per_model = [models[a](samples_per_model[a]) for a in range(nmodels)]

## Stage 5: Compute Estimate

In [ ]:
estimate = fitted(values_per_model)

true_mean = float(bkd.to_numpy(benchmark.ensemble_means()[0, 0]))
print(f"Estimate:  {float(estimate):.6f}")
print(f"True mean: {true_mean:.6f}")
print(f"Error:     {abs(float(estimate) - true_mean):.6f}")

## Compare againt MC

In [ ]:
stat_mc = MultiOutputMean(nqoi, bkd)
stat_mc.set_pilot_quantities(cov_pilot[:1, :1])
mc_est = MCEstimator(stat_mc, costs[:1])
mc_fitted = MCAllocator(mc_est).allocate(remaining)

mc_var  = float(mc_fitted.covariance()[0, 0])
mf_var  = float(fitted.covariance()[0, 0])
print(f"MC std:  {mc_var**0.5:.6f}")
print(f"MF std:  {mf_var**0.5:.6f}")
print(f"Variance reduction: {mc_var / mf_var:.1f}×")